# Evidence-grounded question answering over a rulebook

The pipeline separates retrieval, fusion, answer generation and evaluation. A sparse retriever provides a transparent baseline. A dense channel can be enabled for semantic matching, and reciprocal-rank fusion combines the rankings without assuming comparable similarity scales.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import requests

from nlp_portfolio.rag import (build_sparse_index, retrieve, format_mcq, option_id,
                               reciprocal_rank_fusion)

BASE = "https://raw.githubusercontent.com/ferrazzipietro/dataset-nlp-math/main/data_rag/excercise/"
folder = Path("data/rulebook")
folder.mkdir(parents=True, exist_ok=True)
for filename in ("rules.json", "questions_mcq_student.json"):
    target = folder / filename
    if not target.exists():
        response = requests.get(BASE + filename, timeout=30)
        response.raise_for_status()
        target.write_bytes(response.content)

rules = json.loads((folder / "rules.json").read_text())
questions = json.loads((folder / "questions_mcq_student.json").read_text())
sparse_index = build_sparse_index(rules)
print("rules:", len(rules), "questions:", len(questions))

In [ ]:
def query_text(item):
    options = " ".join(option["text"] for option in item["options"])
    return item["question"] + " " + options

example = questions[0]
sparse_hits = retrieve(query_text(example), sparse_index, top_k=5)
pd.DataFrame(sparse_hits)

## Optional dense retrieval and rank fusion

Dense embeddings help when the question paraphrases the rule. Normalized vectors make the dot product equal cosine similarity. RRF is robust to different score distributions and rewards documents that appear near the top of either list.

In [ ]:
RUN_DENSE = False
dense_rankings = {}
if RUN_DENSE:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    rule_ids = list(rules)
    rule_matrix = encoder.encode([rules[key] for key in rule_ids], normalize_embeddings=True)
    for item in questions:
        q_vector = encoder.encode([query_text(item)], normalize_embeddings=True)[0]
        order = np.argsort(-(rule_matrix @ q_vector))
        dense_rankings[item["id"]] = [rule_ids[i] for i in order]

sparse_rankings = {
    item["id"]: [hit["id"] for hit in retrieve(query_text(item), sparse_index, top_k=20)]
    for item in questions
}
if RUN_DENSE:
    fused_rankings = {key: reciprocal_rank_fusion([sparse_rankings[key], dense_rankings[key]])
                      for key in sparse_rankings}
else:
    fused_rankings = sparse_rankings

## Answer generation with evidence provenance

The model is asked for one option letter and can return `UNKNOWN`. Every prediction stores the retrieved rule identifiers, making failures traceable to retrieval or generation.

In [ ]:
RUN_GENERATION = False
results = []
if RUN_GENERATION:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_id = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    generator = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="auto",
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    for item in questions:
        ids = fused_rankings[item["id"]][:5]
        passages = [{"id": key, "text": rules[key]} for key in ids]
        prompt = format_mcq(item["question"], item["options"], passages)
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=True,
            add_generation_prompt=True, return_dict=True, return_tensors="pt",
        ).to(generator.device)
        with torch.inference_mode():
            output = generator.generate(**inputs, max_new_tokens=8, do_sample=False,
                                        pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results.append({"id": item["id"], "prediction": option_id(raw),
                        "raw_answer": raw, "evidence": ids})
pd.DataFrame(results).head() if results else "generation disabled"

## Evaluation contract

Answer accuracy requires an answer key. Retrieval recall requires independent passage relevance labels. Similarity scores cannot serve as their own ground truth. The code below runs only when a local annotation file is present.

In [ ]:
from nlp_portfolio.retrieval_eval import evaluate_run

qrels_path = folder / "qrels.json"
if qrels_path.exists():
    qrels = json.loads(qrels_path.read_text())
    retrieval_report = evaluate_run(fused_rankings, qrels, k=5)
    print(retrieval_report)
else:
    print("Add qrels.json to report Recall@5, MRR and MAP. No score was guessed.")

if results:
    pd.DataFrame(results)[["id", "prediction"]].to_csv("rulebook_predictions.csv", index=False)

A useful ablation compares sparse, dense and fused rankings on the same qrels. If retrieval recall is high while answer accuracy remains low, prompt or generator errors dominate. If the gold rule is absent from the top five, changing the generator cannot repair the pipeline.